In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import geopandas as gpd
import contextily as ctx
import rasterio
from tqdm import tqdm
import xml.etree.ElementTree as ET
from io import StringIO
import numpy as np
from scipy.spatial import cKDTree
current_dir = os.getcwd()
import sys
sys.path.append(os.path.join(os.path.dirname(current_dir), 'python_libs'))
from jb_2009_corr import jayaram_baker_correlation_2009
from tqdm import tqdm
import scipy as sp

In [2]:
os.path.dirname(current_dir)

'/resnick/groups/enceladus/jyzhao/CAPowerResilience/Analysis'

In [3]:
eq_scenario_id = "bssc2014ssanandreasnsbssbbgc_m7p64_se"
sites_fp = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/eq_handoff_v5/handoff/sites_all.csv"
sites_df = pd.read_csv(sites_fp)
print(f"Number of sites: {len(sites_df)}")

Number of sites: 8899


In [4]:
eq_scenario_dir = f"/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/{eq_scenario_id}"
pga_raster_fp = os.path.join(eq_scenario_dir, "raster", "pga.fit")
with rasterio.open(pga_raster_fp,
    crs="EPSG:4326") as src:
    # Query metadata
    print("Metadata Summary:")
    print(f"Dimensions: {src.width}x{src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    
    # 2. Query and read pixel data (e.g., Band 1)
    for site_id in tqdm(sites_df.index, total=len(sites_df)):
        site_lon, site_lat = sites_df['lon'][site_id], sites_df['lat'][site_id]
        row, col = src.index(site_lon, site_lat)
        # 3. Read the pixel value for the site
        if 0 <= row < src.height and 0 <= col < src.width:
            site_pga = src.read(1)[row, col]
            sites_df.loc[site_id, 'site_pga'] = site_pga
            # print(f"PGA value for the site: {site_pga}")
        else:
            site_pga = 0.0
            sites_df.loc[site_id, 'site_pga'] = site_pga
            # print("Site is outside the raster bounds.")

Metadata Summary:
Dimensions: 810x613
Number of bands: 1
Coordinate Reference System (CRS): None


  0%|          | 0/8899 [00:00<?, ?it/s]

100%|██████████| 8899/8899 [00:04<00:00, 2212.52it/s]


In [5]:
uncertainty_xml_fp = os.path.join(eq_scenario_dir, "uncertainty.xml")
def shakemap_xml_to_df(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Handle the XML namespace
    ns = {"sm": "http://earthquake.usgs.gov/eqcenter/shakemap"}

    # Extract column names, ordered by index
    fields = root.findall("sm:grid_field", ns)
    fields = sorted(fields, key=lambda f: int(f.attrib["index"]))
    columns = [f.attrib["name"] for f in fields]

    # Extract grid data
    grid_data = root.find("sm:grid_data", ns).text

    # Convert whitespace-separated data into DataFrame
    df = pd.read_csv(
        StringIO(grid_data),
        sep=r"\s+",
        header=None,
        names=columns
    )

    return df
uncertainty_df = shakemap_xml_to_df(uncertainty_xml_fp)
uncertainty_gdf = gpd.GeoDataFrame(
    uncertainty_df,
    geometry=gpd.points_from_xy(uncertainty_df['LON'], uncertainty_df['LAT']),
    crs="EPSG:4326"
)

In [6]:
## Nearest n neighbors
n_neighbors = 5
uncertainty_gdf_proj = uncertainty_gdf.to_crs(epsg=3857)
sites_gdf = gpd.GeoDataFrame(
    sites_df,
    geometry=gpd.points_from_xy(sites_df['lon'], sites_df['lat']),
    crs="EPSG:4326"
)
sites_gdf_proj = sites_gdf.to_crs(epsg=3857)
# Extract coordinates
sites_xy = np.column_stack([sites_gdf_proj.geometry.x, sites_gdf_proj.geometry.y])
uncertainty_xy = np.column_stack([uncertainty_gdf_proj.geometry.x, uncertainty_gdf_proj.geometry.y])

tree = cKDTree(uncertainty_xy)
distances, indices = tree.query(sites_xy, k=n_neighbors, workers=-1)

In [7]:
sites_df['GMPE_INTER_STDPGA'] = np.nan
sites_df['GMPE_INTRA_STDPGA'] = np.nan
for i, site_id in tqdm(enumerate(sites_df.index), total=len(sites_df)):
    nearest_indices = indices[i]
    nearest_distances = distances[i]

    weights = 1 / nearest_distances
    weights /= weights.sum()

    # You can now use the weights to compute a weighted average of the nearest neighbors' values
    nearest_values = uncertainty_gdf.iloc[nearest_indices]['GMPE_INTER_STDPGA']  # Replace 'value_column' with the actual column name
    weighted_average = np.sum(weights * nearest_values)
    sites_df.loc[site_id, 'GMPE_INTER_STDPGA'] = weighted_average

    nearest_values = uncertainty_gdf.iloc[nearest_indices]['GMPE_INTRA_STDPGA']  # Replace 'value_column' with the actual column name
    weighted_average = np.sum(weights * nearest_values)
    sites_df.loc[site_id, 'GMPE_INTRA_STDPGA'] = weighted_average


100%|██████████| 8899/8899 [00:10<00:00, 843.85it/s]


In [8]:
print(f"Mean and StdDev GMPE_INTRA_STDPGA: {sites_df['GMPE_INTRA_STDPGA'].mean()}, {sites_df['GMPE_INTRA_STDPGA'].std()}")
print(f"Mean and StdDev GMPE_INTER_STDPGA: {sites_df['GMPE_INTER_STDPGA'].mean()}, {sites_df['GMPE_INTER_STDPGA'].std()}")

Mean and StdDev GMPE_INTRA_STDPGA: 0.5999999999999999, 1.323864756657046e-16
Mean and StdDev GMPE_INTER_STDPGA: 0.29999999999999993, 6.61932378328523e-17


In [9]:
num_gm_samples = 32
sample_seed = 42
print(f"Number of GM samples: {num_gm_samples}")
print(f"Sample seed: {sample_seed}")

Number of GM samples: 32
Sample seed: 42


In [10]:
def sample_mvn(mean, std, corr, k, seed=None):
    mean = np.asarray(mean, dtype=np.float64).reshape(-1)
    std = np.asarray(std, dtype=np.float64).reshape(-1)
    corr = np.asarray(corr, dtype=np.float64)

    # Enforce symmetry to remove floating-point asymmetry
    corr = (corr + corr.T) / 2

    # Cholesky decomposition of correlation matrix
    L = sp.linalg.cholesky(corr, lower=True)

    # Independent standard normal samples
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((len(mean), k))

    # Correlated samples: (n, k)
    X = mean[:, None] + std[:, None] * (L @ Z)
    return X
    

In [11]:
non_zero_sites_idx = sites_df[sites_df['site_pga'] > 0].index
# sample intra-event correlation for non-zero sites
correlations_intra = np.zeros((len(non_zero_sites_idx), len(non_zero_sites_idx)))
for i in tqdm(range(len(non_zero_sites_idx)), total=len(non_zero_sites_idx)):
    for j in range(i + 1, len(non_zero_sites_idx)):
        site_i = non_zero_sites_idx[i]
        site_j = non_zero_sites_idx[j]
        site_i_xy = sites_xy[site_i]
        site_j_xy = sites_xy[site_j]
        h = np.linalg.norm(site_i_xy - site_j_xy)
        rho = jayaram_baker_correlation_2009('PGA', h)
        correlations_intra[i, j] = rho
        correlations_intra[j, i] = rho
np.fill_diagonal(correlations_intra, 1)

100%|██████████| 4650/4650 [01:26<00:00, 53.71it/s] 


In [12]:
intra_res_mean = np.zeros((len(non_zero_sites_idx), 1))
intra_res_std = sites_df.loc[non_zero_sites_idx, 'GMPE_INTRA_STDPGA'].values.reshape(-1, 1)
intra_res_samples = sample_mvn(intra_res_mean.flatten(), intra_res_std.flatten(), correlations_intra, num_gm_samples, seed=sample_seed)

In [13]:
intra_res_samples

array([[ 0.18283025, -0.62399046,  0.45027072, ...,  0.2584926 ,
         1.28498856, -0.24384901],
       [-0.30734564, -0.48826364,  0.36958765, ..., -0.20093102,
         0.09765184,  0.3517334 ],
       [ 0.42673595,  0.47600834, -0.20923504, ..., -0.04783093,
        -1.01240066, -0.86826748],
       ...,
       [ 0.57423014, -0.87353499, -0.0884799 , ...,  0.85521625,
         1.1938436 , -2.24924761],
       [-0.10495083, -0.33102755,  0.57197791, ..., -0.31170137,
        -0.81605973, -0.47385545],
       [-0.06430973, -0.18615286, -0.59596344, ..., -0.34480051,
         0.11038342,  0.2739471 ]], shape=(4650, 32))

In [14]:
inter_res_mean = np.zeros((len(non_zero_sites_idx), 1))
inter_res_std = sites_df.loc[non_zero_sites_idx, 'GMPE_INTER_STDPGA'].values.reshape(-1, 1)
rng = np.random.default_rng(sample_seed+1)

inter_res_samples =np.outer(inter_res_std.flatten(), rng.standard_normal(num_gm_samples))
print(inter_res_samples.shape)
inter_res_samples

(4650, 32)


array([[ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777],
       [ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777],
       [ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777],
       ...,
       [ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777],
       [ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777],
       [ 0.07326885,  0.2034535 , -0.17565881, ..., -0.61518281,
         0.16659111,  0.11888777]], shape=(4650, 32))

In [15]:
sites_gm_samples = np.zeros((num_gm_samples, len(sites_df)))
sites_gm_samples[:, non_zero_sites_idx] = sites_df.loc[non_zero_sites_idx, 'site_pga'].values.reshape(1, -1)
sites_gm_samples[:, non_zero_sites_idx] = np.log(sites_gm_samples[:, non_zero_sites_idx])
sites_gm_samples[:, non_zero_sites_idx] += inter_res_samples.T # The inter-event residuals are lognormal distributed
sites_gm_samples[:, non_zero_sites_idx] += intra_res_samples.T # The intra-event residuals are lognormal distributed    
sites_gm_samples[:, non_zero_sites_idx] = np.exp(sites_gm_samples[:, non_zero_sites_idx])
sites_gm_samples = sites_gm_samples / 100 # convert from pctg to g

In [16]:
sites_gm_samples_df = pd.DataFrame(sites_gm_samples, columns=sites_df.index)
sites_gm_samples_df.to_csv(os.path.join(
    eq_scenario_dir, 'sites_gm_samples.csv'
), index=False)

In [17]:
sites_gm_samples = np.zeros((num_gm_samples, len(sites_df)))
sites_gm_samples[:, non_zero_sites_idx] = sites_df.loc[non_zero_sites_idx, 'site_pga'].values.reshape(1, -1)
sites_gm_samples[:, non_zero_sites_idx] = np.log(sites_gm_samples[:, non_zero_sites_idx])
sites_gm_samples[:, non_zero_sites_idx] += inter_res_samples.T # The inter-event residuals are lognormal distributed
sites_gm_samples[:, non_zero_sites_idx] += intra_res_samples.T # The intra-event residuals are lognormal distributed 
sites_gm_samples[:, non_zero_sites_idx] = np.exp(sites_gm_samples[:, non_zero_sites_idx])
sites_gm_samples[:, non_zero_sites_idx].max(axis=1)

array([318.34194088, 382.14271809, 188.93278717, 270.06208032,
       196.39861864, 763.25654171, 240.41972706, 269.08798489,
       227.79724232, 497.12529316, 431.76516863, 330.09810303,
       332.42491629, 359.26326279, 238.18588604, 276.71579917,
       446.28622607, 402.09200876, 277.42999649, 279.01759905,
       533.72728207, 193.2549262 , 249.5447093 , 291.49422258,
       140.16486043, 452.32603815, 370.62073928, 253.17196673,
       315.77314627, 128.76104444, 334.87208226, 267.11517978])